In [1]:
from collections.abc import Sequence
from typing import cast

import matplotlib
import matplotlib.pyplot as plt
import torch
import torchvision.transforms.functional as F
import torchvision
from PIL import Image
from torchvision.utils import make_grid

import sys,os
from pathlib import Path
sys.path.append(str(Path(os.getcwd()).resolve().parent.parent))

from shimmer.modules.global_workspace import (
    GlobalWorkspaceFusion,
)

from shimmer_metaworld import DEBUG_MODE, PROJECT_DIR,LOGGER
from shimmer_metaworld.config import load_config
from shimmer_metaworld.logging import get_pil_image, batch_to_device
from metaworld_dataset import (
    MetaworldDataModule,
    DomainDesc,
    get_default_domains,
)
from shimmer_metaworld.modules.domains import load_pretrained_domains

matplotlib.use("Agg")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import repsim

**Loading and pickling model variables**

In [ ]:
import pickle
#clip -10,10 to remove weird values but the just mean stdv norm
act_no_clip = {
    "gw_ckpt" : "rddwyo3s",
    "ckpt_epoch" : "176"
}
act_no_clip_clip = {
    "gw_ckpt" : "puo9ybm9",
    "ckpt_epoch" : "174"
}
act = {
    "gw_ckpt" : "at9euav0",
    "ckpt_epoch" : "145"
}
current_model = "act"
if current_model == "act":
    gw_ckpt = act_no_clip["gw_ckpt"]
    ckpt_epoch = act_no_clip["ckpt_epoch"]
else:
    gw_ckpt = no_act["gw_ckpt"]
    ckpt_epoch = no_act["ckpt_epoch"]


def image_grid_from_v_tensor(
    samples: Sequence[torch.Tensor],
    _: int,
    ncols: int,
) -> Image:
    image = make_grid(samples[0], nrow=ncols, pad_value=1).detach()
    return F.to_pil_image(image)


debug_mode = DEBUG_MODE
extra_config_files = ["train_gw.yaml"]
argv = []

LOGGER.debug(f"Debug mode: {debug_mode}")

config = load_config(
    PROJECT_DIR / "shimmer_metaworld"/ "config_template",
    load_files=extra_config_files,
    debug_mode=debug_mode,
    log_config=False,
    argv=argv,
)
print(gw_ckpt)
#seed_everything(config.seed, workers=True)

domain_classes = get_default_domains(
    {domain.domain_type.kind.value for domain in config.domains}
)
print(config.domains)
domain_modules, gw_encoders, gw_decoders = load_pretrained_domains(
    config.domains,
    config.global_workspace.latent_dim,
    config.global_workspace.encoders.hidden_dim,
    config.global_workspace.encoders.n_layers,
    config.global_workspace.decoders.hidden_dim,
    config.global_workspace.decoders.n_layers,
    is_linear=config.global_workspace.linear_domains,
    bias=config.global_workspace.linear_domains_use_bias,
)



ckpt_path = f'/mnt/datashare/yelhelw/checkpoints/shimmer-meta-{gw_ckpt}/epoch={ckpt_epoch}.ckpt'

domain_module = GlobalWorkspaceFusion.load_from_checkpoint(ckpt_path, domain_mods=domain_modules,
    gw_encoders=gw_encoders,
    gw_decoders=gw_decoders)
domain_module.eval().freeze()

domain_module.to(device)


with open(f"models/{current_model}/domain_module.pkl","wb") as f:
    pickle.dump(domain_module,f)
with open(f"models/{current_model}/domain_modules.pkl","wb") as f:
    pickle.dump(domain_modules,f)



In [ ]:
#Load data
domain_classes = get_default_domains(["v_latents","attr","act"])

print(config.domain_proportions)
data_module = MetaworldDataModule(
        '/mnt/datashare/yelhelw/dataset',
        domain_classes,
        config.domain_proportions,
        batch_size=config.training.batch_size,
        num_workers=config.training.num_workers,
        seed=config.seed,
        ood_seed=config.ood_seed,
        domain_args=config.domain_data_args,
    )



#with open(f"models/{current_model}/data_module.pkl","wb") as f:
#    pickle.dump(data_module,f)


__Latent UMAP structures__

In [2]:
from collections.abc import Mapping
from shimmer.modules.selection import FixedSharedSelection
from matplotlib.colors import ListedColormap
from tqdm import tqdm


def to_device(data: torch.Tensor | Mapping[str, torch.Tensor] | list, device: str):
    """Put the data Tensor or list on the device (GPU or CPU)"""
    if isinstance(data, torch.Tensor):
        return data.to(device)
    elif isinstance(data, list):
        return [value.to(device) for value in data]
    elif isinstance(data, Mapping):
        return {name: to_device(value, device) for name, value in data.items()}
    else:
        raise TypeError(f"Unsupported type: {type(data)}")


<p align="center">
    General Embeddings
</p>

In [2]:
import numpy as np
labels = np.load('/mnt/datashare/yelhelw/complex_dataset/actions_val.npy', mmap_mode="r")
labels_train = np.clip(np.load('/mnt/datashare/yelhelw/complex_dataset/actions_train.npy', mmap_mode="r"), -10, 10)
all_labels = np.repeat(labels, 3, axis=0)
attributes = np.load('/mnt/datashare/yelhelw/complex_dataset/attributes_val.npy', mmap_mode="r")
wall = attributes[:,8]
ball = attributes[:,6]
goal = attributes[:,12]

print(ball)
modality_names = {0: 'Vision (v)', 1: 'Attributes (attr)', 2: 'Actions (act)'}


labels_norm = labels.copy()

#labels_norm = labels.copy()
action_mean = np.mean(labels_train,axis=0)
action_stdv = np.std(labels_train,axis=0)
for x in range(4):
    labels_norm[:,x] = (labels_norm[:,x]-action_mean[x])/ action_stdv[x]
#    labels_norm[:,x] = np.clip(labels_norm[:,x],-1,1)
x_disp = labels_norm[:, 0]
y_disp = labels_norm[:, 1]
z_disp = labels_norm[:, 2]
gripper = labels_norm[:, 3]

print("Transforming latent vectors...")


[0.0226834  0.02621185 0.02396803 ... 0.02525873 0.02236027 0.04183513]
Transforming latent vectors...


In [4]:

plt.figure(figsize=(20, 20))
plt.hist(np.clip(labels[:,1], -1, 1), bins=50)
plt.savefig(f'graphs/action_dist/y_brut.png')
plt.close()
plt.figure(figsize=(20, 20))
plt.hist(y_disp, bins=50)
plt.savefig(f'graphs/action_dist/y_norm.png')
plt.close()

In [7]:
# Analyze 2D histogram to find interesting action combinations
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# X vs Y displacement
h1, xedges, yedges, img1 = axes[0, 0].hist2d(x_disp, y_disp, bins=30, cmap='viridis')
axes[0, 0].set_xlabel('X displacement (right-left)')
axes[0, 0].set_ylabel('Y displacement (front-back)')
axes[0, 0].set_title('X vs Y Action Distribution')
plt.colorbar(img1, ax=axes[0, 0])

# X vs Z displacement  
h2, _, _, img2 = axes[0, 1].hist2d(y_disp, z_disp, bins=30, cmap='viridis')
axes[0, 1].set_xlabel('X displacement (right-left)')
axes[0, 1].set_ylabel('Z displacement (up-down)')
axes[0, 1].set_title('X vs Z Action Distribution')
plt.colorbar(img2, ax=axes[0, 1])

# X vs Gripper
h3, _, _, img3 = axes[1, 0].hist2d(y_disp, gripper, bins=30, cmap='viridis')
axes[1, 0].set_xlabel('X displacement (right-left)')
axes[1, 0].set_ylabel('Gripper')
axes[1, 0].set_title('X vs Gripper Action Distribution')
plt.colorbar(img3, ax=axes[1, 0])

# Z vs Gripper
h4, _, _, img4 = axes[1, 1].hist2d(z_disp, gripper, bins=30, cmap='viridis')
axes[1, 1].set_xlabel('Z displacement (up-down)')
axes[1, 1].set_ylabel('Gripper')
axes[1, 1].set_title('Z vs Gripper Action Distribution')
plt.colorbar(img4, ax=axes[1, 1])

plt.tight_layout()
plt.savefig(f'graphs/action_dist/action_distributions_complex_1.png')
plt.show()

# Find peak action combinations from 2D histograms
print("=== Most Common Action Combinations ===\n")

# For X-Y: find bins with highest counts
flat_idx = np.argsort(h1.flatten())[::-1][:10]
print("Top X-Y combinations (count, x_center, y_center):")
for idx in flat_idx:
    i, j = np.unravel_index(idx, h1.shape)
    x_center = (xedges[i] + xedges[i+1]) / 2
    y_center = (yedges[j] + yedges[j+1]) / 2
    print(f"  Count: {int(h1[i,j]):5d} | X: {x_center:+.2f} | Y: {y_center:+.2f}")

print("\n=== Suggested Action Filters for UMAP ===")
print("Based on distribution peaks:")

# Identify distinct action modes
action_modes = {
    'push_forward': (np.abs(x_disp) < 0.3) & (y_disp > 0.5),
    'push_left': (x_disp < -0.5) & (np.abs(y_disp) < 0.3),
    'push_right': (x_disp > 0.5) & (np.abs(y_disp) < 0.3),
    'diagonal_left_forward': (x_disp < -0.3) & (y_disp > 0.3),
    'diagonal_right_forward': (x_disp > 0.3) & (y_disp > 0.3),
    'stationary': (np.abs(x_disp) < 0.2) & (np.abs(y_disp) < 0.2),
    'gripper_active': gripper > 0.5,
    'gripper_closed': gripper < 0.2,
}

for name, mask in action_modes.items():
    count = mask.sum()
    if count > 100:  # Only show modes with enough samples
        print(f"  {name}: {count} samples ({100*count/len(x_disp):.1f}%)")

=== Most Common Action Combinations ===

Top X-Y combinations (count, x_center, y_center):
  Count:  7364 | X: -0.09 | Y: -0.44
  Count:  7213 | X: +0.11 | Y: -0.10
  Count:  5899 | X: +0.11 | Y: -0.44
  Count:  4884 | X: +0.11 | Y: -0.27
  Count:  3239 | X: -0.09 | Y: -0.27
  Count:  3189 | X: +0.11 | Y: +3.87
  Count:  3089 | X: -2.93 | Y: -0.44
  Count:  2928 | X: +2.95 | Y: -0.44
  Count:  1232 | X: +0.31 | Y: -0.27
  Count:  1198 | X: -0.09 | Y: -0.10

=== Suggested Action Filters for UMAP ===
Based on distribution peaks:
  push_forward: 4899 samples (8.2%)
  push_left: 1189 samples (2.0%)
  push_right: 1306 samples (2.2%)
  diagonal_left_forward: 1279 samples (2.1%)
  diagonal_right_forward: 1318 samples (2.2%)
  stationary: 11124 samples (18.5%)
  gripper_active: 21432 samples (35.7%)
  gripper_closed: 25051 samples (41.7%)


In [6]:
# Analyze 2D histogram with UNNORMALIZED action values
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Unnormalized action values
x_raw = np.clip(labels[:, 0],-1,1)
y_raw = np.clip(labels[:, 1],-1,1)
z_raw = np.clip(labels[:, 2],-1,1)
gripper_raw = np.clip(labels[:, 3],-1,1)

# X vs Y displacement (unnormalized)
h1, xedges, yedges, img1 = axes[0, 0].hist2d(x_raw, y_raw, bins=30, cmap='viridis')
axes[0, 0].set_xlabel('X displacement (raw)')
axes[0, 0].set_ylabel('Y displacement (raw)')
axes[0, 0].set_title('X vs Y Action Distribution (Unnormalized)')
plt.colorbar(img1, ax=axes[0, 0])

# X vs Z displacement (unnormalized)
h2, _, _, img2 = axes[0, 1].hist2d(y_raw, z_raw, bins=30, cmap='viridis')
axes[0, 1].set_xlabel('Y displacement (raw)')
axes[0, 1].set_ylabel('Z displacement (raw)')
axes[0, 1].set_title('Y vs Z Action Distribution (Unnormalized)')
plt.colorbar(img2, ax=axes[0, 1])

# Y vs Gripper (unnormalized)
h3, _, _, img3 = axes[1, 0].hist2d(y_raw, gripper_raw, bins=30, cmap='viridis')
axes[1, 0].set_xlabel('Y displacement (raw)')
axes[1, 0].set_ylabel('Gripper (raw)')
axes[1, 0].set_title('Y vs Gripper Action Distribution (Unnormalized)')
plt.colorbar(img3, ax=axes[1, 0])

# Z vs Gripper (unnormalized)
h4, _, _, img4 = axes[1, 1].hist2d(z_raw, gripper_raw, bins=30, cmap='viridis')
axes[1, 1].set_xlabel('Z displacement (raw)')
axes[1, 1].set_ylabel('Gripper (raw)')
axes[1, 1].set_title('Z vs Gripper Action Distribution (Unnormalized)')
plt.colorbar(img4, ax=axes[1, 1])

plt.tight_layout()
plt.savefig(f'graphs/act/action_distributions_complex_unnormalized.png')
plt.show()

# Print raw value ranges
print("=== Raw Action Value Ranges ===")
print(f"X: [{x_raw.min():.3f}, {x_raw.max():.3f}]")
print(f"Y: [{y_raw.min():.3f}, {y_raw.max():.3f}]")
print(f"Z: [{z_raw.min():.3f}, {z_raw.max():.3f}]")
print(f"Gripper: [{gripper_raw.min():.3f}, {gripper_raw.max():.3f}]")

=== Raw Action Value Ranges ===
X: [-1.000, 1.000]
Y: [-1.000, 1.000]
Z: [-1.000, 1.000]
Gripper: [-1.000, 1.000]


In [ ]:
# 3D and 4D visualization of action distributions
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.colors as mcolors

# =============================================================================
# 3D Scatter Plot: X, Y, Z with Gripper as color
# =============================================================================
fig = plt.figure(figsize=(16, 6))

# Subsample for visualization (too many points makes it slow)
n_samples = min(5000, len(x_disp))
idx = np.random.choice(len(x_disp), n_samples, replace=False)

# 3D scatter with gripper as color
ax1 = fig.add_subplot(121, projection='3d')
scatter1 = ax1.scatter(x_disp[idx], y_disp[idx], z_disp[idx], 
                       c=gripper[idx], cmap='plasma', s=5, alpha=0.6)
ax1.set_xlabel('X displacement')
ax1.set_ylabel('Y displacement')
ax1.set_zlabel('Z displacement')
ax1.set_title('3D Action Space (color = Gripper)')
plt.colorbar(scatter1, ax=ax1, label='Gripper', shrink=0.6)

# 3D scatter with density estimation as color
from scipy.stats import gaussian_kde
xyz = np.vstack([x_disp[idx], y_disp[idx], z_disp[idx]])
try:
    density = gaussian_kde(xyz)(xyz)
except:
    density = np.ones(len(idx))  # fallback if KDE fails

ax2 = fig.add_subplot(122, projection='3d')
scatter2 = ax2.scatter(x_disp[idx], y_disp[idx], z_disp[idx],
                       c=density, cmap='hot', s=5, alpha=0.6)
ax2.set_xlabel('X displacement')
ax2.set_ylabel('Y displacement')
ax2.set_zlabel('Z displacement')
ax2.set_title('3D Action Space (color = Density)')
plt.colorbar(scatter2, ax=ax2, label='Density', shrink=0.6)

plt.tight_layout()
plt.savefig(f'graphs/{current_model}/action_distributions_3d.png', dpi=150)
plt.show()

# =============================================================================
# Cluster Centers - Most Common Action Combinations
# =============================================================================
print("\n=== Most Common Action Combinations (Cluster Centers) ===")
cluster_centers = pd.DataFrame(kmeans.cluster_centers_, 
                               columns=['X', 'Y', 'Z', 'Gripper'])
cluster_counts = action_df['Cluster'].value_counts().sort_index()
cluster_centers['Count'] = cluster_counts.values
cluster_centers['Percentage'] = 100 * cluster_centers['Count'] / len(action_df)
cluster_centers = cluster_centers.sort_values('Count', ascending=False)

print(cluster_centers.to_string(float_format='{:.3f}'.format))

# =============================================================================
# 3D Histogram (binned voxel plot)
# =============================================================================
fig = plt.figure(figsize=(14, 6))

# Create 3D histogram by binning
n_bins = 10
hist_3d, edges = np.histogramdd(np.column_stack([x_disp, y_disp, z_disp]), 
                                 bins=n_bins, range=[[-1, 1], [-1, 1], [-1, 1]])

# Get bin centers
x_centers = (edges[0][:-1] + edges[0][1:]) / 2
y_centers = (edges[1][:-1] + edges[1][1:]) / 2
z_centers = (edges[2][:-1] + edges[2][1:]) / 2

# Create meshgrid for voxel centers
xx, yy, zz = np.meshgrid(x_centers, y_centers, z_centers, indexing='ij')

# Flatten and filter low-count voxels
threshold = np.percentile(hist_3d[hist_3d > 0], 50)  # Show top 50%
mask = hist_3d > threshold

ax1 = fig.add_subplot(121, projection='3d')
scatter = ax1.scatter(xx[mask], yy[mask], zz[mask], 
                      c=hist_3d[mask], cmap='hot', 
                      s=hist_3d[mask] / hist_3d.max() * 200, alpha=0.7)
ax1.set_xlabel('X displacement')
ax1.set_ylabel('Y displacement')
ax1.set_zlabel('Z displacement')
ax1.set_title('3D Histogram (bubble size & color = count)')
plt.colorbar(scatter, ax=ax1, label='Count', shrink=0.6)

# 4D: show top bins with gripper info
hist_4d, edges_4d = np.histogramdd(
    np.column_stack([x_disp, y_disp, z_disp, gripper]),
    bins=8, range=[[-1, 1], [-1, 1], [-1, 1], [-1, 1]]
)

# Find top 20 most common 4D bins
flat_idx = np.argsort(hist_4d.flatten())[::-1][:20]
top_bins = np.array(np.unravel_index(flat_idx, hist_4d.shape)).T

ax2 = fig.add_subplot(122, projection='3d')
print("\n=== Top 20 Most Common 4D Action Combinations ===")
print(f"{'Rank':<5} {'X':>8} {'Y':>8} {'Z':>8} {'Grip':>8} {'Count':>8}")
print("-" * 45)

for rank, (i, j, k, l) in enumerate(top_bins):
    x_c = (edges_4d[0][i] + edges_4d[0][i+1]) / 2
    y_c = (edges_4d[1][j] + edges_4d[1][j+1]) / 2
    z_c = (edges_4d[2][k] + edges_4d[2][k+1]) / 2
    g_c = (edges_4d[3][l] + edges_4d[3][l+1]) / 2
    count = int(hist_4d[i, j, k, l])
    print(f"{rank+1:<5} {x_c:>8.2f} {y_c:>8.2f} {z_c:>8.2f} {g_c:>8.2f} {count:>8}")
    
    # Plot in 3D with gripper as color
    ax2.scatter([x_c], [y_c], [z_c], c=[g_c], cmap='coolwarm', 
                s=count/hist_4d.max()*500, vmin=-1, vmax=1, 
                alpha=0.8, edgecolors='black', linewidth=0.5)

ax2.set_xlabel('X displacement')
ax2.set_ylabel('Y displacement')
ax2.set_zlabel('Z displacement')
ax2.set_title('Top 20 Action Combos (size=count, color=gripper)')
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(-1, 1))
plt.colorbar(sm, ax=ax2, label='Gripper', shrink=0.6)

plt.tight_layout()
plt.savefig(f'graphs/{current_model}/action_distributions_3d_4d_histogram.png', dpi=150)
plt.show()